# Do Temperature, Drought, and Wind Predict Wildfire Severity

---

## 1. Observation

Wildfires vary dramatically in size. Some burn a fraction of a hectare; others consume hundreds. This variation happens even within the same park, the same month, sometimes the same week. The question is: **is that variation random, or is it encoded in the environmental conditions before the fire starts?**

This project uses 517 fire records from Montesinho Natural Park, Portugal to investigate whether measurable weather and drought conditions can predict wildfire severity. We approach this in two deliberate stages that build on each other:

- **Stage 1 — Discovery (Clustering):** We ask the data a simple question first: *do fires naturally group into distinct environmental types, without any labels or assumptions?* If yes, and if those groups differ in fire size, it tells us that conditions carry real signal, which justifies trying to predict fire size directly.
- **Stage 2 — Confirmation (Classification):** Armed with that evidence, we build classifiers to predict whether a fire will be large or small, and test whether a model can detect the same patterns the clustering revealed.

---

## 2. Research Question

**Do Temperature, Drought, and Wind Predict Wildfire Severity**

---

## 3. Hypotheses

**H1:** Higher temperature is associated with larger fires.

**H2:** Higher Drought Code (DC) is associated with larger fires.

**H3:** Higher wind speed is associated with larger fires.

**H4:** A flexible nonlinear model (Random Forest) will outperform simpler linear models. The relationships between conditions and fire size are complex and interactive, not purely linear. 

---

## 4. Data 

The dataset records 517 fires in Montesinho Park with weather conditions and burned area in hectares.

### Data Preparation
- **Log-transform area:** `log(area + 1)` compresses the extreme skew (max raw area = 1090 ha)
- **Classification target:** fires above the median log-area are labelled *large*, below are *small* — giving balanced classes
- **Clustering uses all 517 fires** including zero-area; classification uses only the 270 that spread

---

In [1]:
# ─────────────────────────────────────────────────────────────────────────
# IMPORTS
# Two core packages:
#   plotly     — all 5 interactive visuals
#   scikit-learn — KMeans, PCA, RandomForest, evaluation metrics
# ─────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

#  Wildfire colour palette 
C_LOW    = '#F59E0B'   # amber       — Low Danger cluster / Small Fire
C_MOD    = '#EA580C'   # orange      — Moderate Danger cluster
C_HIGH   = '#DC2626'   # red         — High Danger cluster / Large Fire
C_GRAY   = '#9CA3AF'   # gray        — baselines, random chance
C_TITLE  = '#2C1810'   # dark brown  — titles and axes

# Cluster colour map keyed by stable integer label (0=low,1=mod,2=high)
CLUST_COLORS = {0: C_LOW, 1: C_MOD, 2: C_HIGH}
CLUST_NAMES = {0: 'Cluster 1', 1: 'Cluster 2', 2: 'Cluster 3'}

In [2]:
# DATA WRANGLING

# Full Dataset
df_full = pd.read_csv('forestfires.csv').copy()
df_full['fire_id']  = np.arange(1, len(df_full) + 1)
df_full['log_area'] = np.log1p(df_full['area'])

#  Clustering feature matrix 
cluster_features = ['FFMC','DMC','DC','ISI','temp','RH','wind','rain','area']
data_clust = df_full[cluster_features].copy()
data_clust['log_area'] = np.log1p(data_clust['area'])
data_clust = data_clust.drop(columns='area')
FINAL_FEATURES = ['FFMC','DMC','DC','ISI','temp','RH','wind','rain','log_area']

# StandardScaler: z-score normalisation so no single feature dominates by scale
scaler_clust = StandardScaler()
X_scaled_arr = scaler_clust.fit_transform(data_clust)
X_scaled_df  = pd.DataFrame(X_scaled_arr, columns=FINAL_FEATURES, index=df_full.index)

# PCA: reduce 9 dimensions to 2 for visualisation only — clustering uses full space
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_df)
print(f'PCA explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%  PC2={pca.explained_variance_ratio_[1]*100:.1f}%')

#  Filtered dataset — used for classification (270 fires that spread) 
df = df_full[df_full['area'] > 0].copy()
df['log_area']    = np.log1p(df['area'])
median_log        = df['log_area'].median()
df['large_fire']  = (df['log_area'] > median_log).astype(int)
month_counts      = df['month'].value_counts()
rare_months       = month_counts[month_counts < 10].index.tolist()
df['month_grouped'] = df['month'].apply(lambda x: 'other' if x in rare_months else x)
df['temp_group']    = pd.qcut(df['temp'], q=4, labels=['Low','Medium','High','Very High'])

print(f'\nFull dataset: {len(df_full)} fires')
print(f'Fires that spread: {len(df)} | Large: {df["large_fire"].sum()} | Small: {(df["large_fire"]==0).sum()}')
print(f'Median log-area threshold: {median_log:.3f}')

PCA explained variance: PC1=31.7%  PC2=14.5%

Full dataset: 517 fires
Fires that spread: 270 | Large: 135 | Small: 135
Median log-area threshold: 1.997


In [3]:

# CLUSTERING HELPERS

def fit_clusters(weight=1.0, k=3):
    X_w = X_scaled_df.copy()
    X_w['DC']   *= weight
    X_w['temp'] *= weight
    X_w['wind'] *= weight
    labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X_w)
    out = df_full.copy()
    out['log_area']    = np.log1p(out['area'])
    out['cluster_raw'] = labels
    out['PC1']         = X_pca[:, 0]
    out['PC2']         = X_pca[:, 1]
    order = out.groupby('cluster_raw')['log_area'].mean().sort_values()
    remap = {old: new for new, old in enumerate(order.index)}
    out['cluster']      = out['cluster_raw'].map(remap)
    out['danger_label'] = out['cluster'].map(CLUST_NAMES)
    return out


def ellipse_points(x, y, n=200, scale=2.0):
    x = np.asarray(x); y = np.asarray(y)
    if len(x) == 1:
        t = np.linspace(0, 2*np.pi, n)
        return x[0] + 0.2*np.cos(t), y[0] + 0.2*np.sin(t)
    if len(x) == 2:
        cx, cy = np.mean(x), np.mean(y)
        dx = max(abs(x[1]-x[0]), 0.2); dy = max(abs(y[1]-y[0]), 0.2)
        t  = np.linspace(0, 2*np.pi, n)
        return cx + dx*np.cos(t), cy + dy*np.sin(t)
    cov  = np.cov(x, y) + np.eye(2)*1e-6
    mean = np.array([np.mean(x), np.mean(y)])
    eigvals, eigvecs = np.linalg.eigh(cov)
    order   = eigvals.argsort()[::-1]
    eigvals = eigvals[order]; eigvecs = eigvecs[:, order]
    t       = np.linspace(0, 2*np.pi, n)
    circle  = np.array([np.cos(t), np.sin(t)])
    ellipse = eigvecs @ np.diag(np.sqrt(np.maximum(eigvals, 1e-6)) * scale) @ circle
    return (ellipse.T + mean)[:, 0], (ellipse.T + mean)[:, 1]


WEIGHTS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
K = 3

# Visual 1 - Basic Scatter Plots

In [4]:
import plotly.graph_objects as go
import plotly.express as px

# Features
FEATURES = [
    ('temp', 'Temperature (°C)', '#DC2626'),
    ('DC',   'Drought Code (DC)', '#EA580C'),
    ('wind', 'Wind Speed (km/h)', '#F59E0B')
]

fig = go.Figure()


# Build traces
for i, (feat, label, color) in enumerate(FEATURES):

    scatter = px.scatter(
        df,
        x=feat,
        y='log_area',
        opacity=0.6,
        trendline='ols'
    )

    for t in scatter.data:
        t.visible = (i == 0)
        t.marker.color = color
        t.name = label
        fig.add_trace(t)

# Buttons
traces_per_feature = len(px.scatter(df, x='temp', y='log_area', trendline='ols').data)

buttons = []

for i, (feat, label, color) in enumerate(FEATURES):

    visibility = []

    for j in range(len(FEATURES)):
        visibility += [j == i] * traces_per_feature

    buttons.append(dict(
        label=label,
        method="update",
        args=[
            {"visible": visibility},
            {
                "title.text": f"<b>{label} vs Burned Area</b>",
                "title.font.color": "#2C1810",
                "xaxis.title.text": label,
                "xaxis.title.font.color": "#2C1810",
                "yaxis.title.text": "Log Burned Area",
                "yaxis.title.font.color": "#2C1810",
            }
        ]
    ))

# Layout 
fig.update_layout(
    title=dict(
        text="<b>Wildfire Drivers vs Burned Area</b>",
        x=0.5,
        y=0.98,
        font=dict(size=18)
    ),
    template="plotly_white",
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=700,
    width=900,
    margin=dict(t=160, l=70, r=30, b=70),
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.5,
        y=1.22,
        xanchor="center",
        yanchor="top",
        buttons=buttons,
        bgcolor="white",
        bordercolor="#EA580C",
        font=dict(size=12, color="#2C1810")
    )],
    font=dict(family="Georgia, serif", size=12, color="#2C1810"),
    xaxis=dict(
        title=FEATURES[0][1],
        gridcolor="#F0E8E0",
        linecolor="#D4C4B8",
        zerolinecolor="#D4C4B8"
    ),
    yaxis=dict(
        title="Log Burned Area",
        gridcolor="#F0E8E0",
        linecolor="#D4C4B8",
        zerolinecolor="#D4C4B8"
    ),
    showlegend=False
)

---
## 5. Stage 1 — Discovery: Do Fires Cluster by Environmental Conditions?

Before building any predictive model, we ask a deliberately simple question: **if we ignore fire size entirely, do the environmental conditions alone produce meaningful groups?**


### Visual 1 - KMeans Cluster Plot (Weight Slider)
Drag the slider to change the feature weight. Hover over any point for the full environmental record of that fire.

In [5]:

# VISUAL 1 — KMeans cluster plot with weight slider


def build_cluster_traces(clustered):
    """Build scatter + ellipse traces for one weight setting."""
    traces = []
    for cl in sorted(clustered['cluster'].unique()):
        sub   = clustered[clustered['cluster'] == cl].copy()
        color = CLUST_COLORS[cl]
        name  = CLUST_NAMES[cl]
        customdata = np.stack([
            sub['fire_id'], sub['month'], sub['day'],
            sub['temp'], sub['RH'], sub['wind'], sub['rain'],
            sub['FFMC'], sub['DMC'], sub['DC'], sub['ISI'],
            sub['area'], sub['log_area'], sub['cluster']
        ], axis=-1)
        traces.append(go.Scatter(
            x=sub['PC1'], y=sub['PC2'], mode='markers',
            name=name, legendgroup=name, showlegend=True,
            marker=dict(size=9, color=color, line=dict(width=0.5, color='white')),
            customdata=customdata,
            hovertemplate=(
                '<b>Fire %{customdata[0]}</b>  —  <b>' + name + '</b><br>'
                'Month: %{customdata[1]}  Day: %{customdata[2]}<br>'
                'Temp: %{customdata[3]}°C  RH: %{customdata[4]}%  Wind: %{customdata[5]} km/h<br>'
                'FFMC: %{customdata[7]}  DMC: %{customdata[8]}  DC: %{customdata[9]}  ISI: %{customdata[10]}<br>'
                'Burned area: %{customdata[11]} ha  (log: %{customdata[12]:.3f})'
                '<extra></extra>'
            )
        ))
        ex, ey = ellipse_points(sub['PC1'].values, sub['PC2'].values, scale=2.0)
        traces.append(go.Scatter(
            x=ex, y=ey, mode='lines', name=f'{name} boundary',
            legendgroup=name, showlegend=False,
            line=dict(color=color, width=2, dash='dot'), hoverinfo='skip'
        ))
    return traces

frames = [go.Frame(name=str(w), data=build_cluster_traces(fit_clusters(w, K))) for w in WEIGHTS]
fig1   = go.Figure(data=build_cluster_traces(fit_clusters(WEIGHTS[0], K)), frames=frames)

steps = [dict(
    method='animate',
    args=[[str(w)], {'mode':'immediate','frame':{'duration':0,'redraw':True},'transition':{'duration':0}}],
    label=str(w)
) for w in WEIGHTS]

fig1.update_layout(
    title=dict(text='<b>Visual 1:</b> Forest Fire Clusters in PCA Space — Drag Slider to Adjust Feature Weight',
               font=dict(size=14, color=C_TITLE)),
    xaxis_title='PC1 — dominant axis of environmental variation',
    yaxis_title='PC2 — secondary axis',
    legend_title='Cluster #', width=1000, height=700,
    template='plotly_white',
    paper_bgcolor='white',
    plot_bgcolor='#FFF8F5',
    font=dict(family='Georgia, serif', size=12, color='#2C1810'),
    sliders=[{'active':0,'currentvalue':{'prefix':'Feature weight: '},'pad':{'t':50},'steps':steps}],
    annotations=[
        dict(text='🔍 At weight ≥ 2.0, three distinct danger groups emerge clearly',
             x=0.01, y=0.99, xref='paper', yref='paper', showarrow=False,
             font=dict(size=11, color=C_TITLE), bgcolor='rgba(255,248,245,0.95)',
             bordercolor=C_TITLE, borderwidth=1)
    ]
)
fig1.show()

### Cluster Summary 
The three clusters are characterised below. Crucially, these descriptions come entirely from environmental features — fire size played no role in forming the groups.

In [6]:
# CLUSTER SUMMARY — weight = 2.0 (chosen as the clearest separation)─
CHOSEN_WEIGHT = 2.0
clustered     = fit_clusters(weight=CHOSEN_WEIGHT, k=K)

summary_vars  = ['FFMC','DMC','DC','ISI','temp','RH','wind','rain','area','log_area']
clust_means   = clustered.groupby('cluster')[summary_vars].mean().round(2)
clust_sizes   = clustered['cluster'].value_counts().sort_index()

print('═'*60)
print('  CLUSTER PROFILES  (weight = 2.0)')
print('═'*60)
for cl in [0, 1, 2]:
    row = clust_means.loc[cl]
    print(f'\n  {CLUST_NAMES[cl]}  (n={clust_sizes[cl]})')
    print(f'    Temp: {row["temp"]}°C  |  DC: {row["DC"]}  |  DMC: {row["DMC"]}')
    print(f'    Avg area: {row["area"]} ha  |  Avg log-area: {row["log_area"]}')
print('\n  Full means:')
print(clust_means.rename(index=CLUST_NAMES))

════════════════════════════════════════════════════════════
  CLUSTER PROFILES  (weight = 2.0)
════════════════════════════════════════════════════════════

  Cluster 1  (n=107)
    Temp: 11.76°C  |  DC: 110.25  |  DMC: 31.09
    Avg area: 5.98 ha  |  Avg log-area: 0.99

  Cluster 2  (n=147)
    Temp: 18.49°C  |  DC: 646.28  |  DMC: 130.45
    Avg area: 9.86 ha  |  Avg log-area: 1.09

  Cluster 3  (n=263)
    Temp: 22.01°C  |  DC: 671.05  |  DMC: 132.39
    Avg area: 17.31 ha  |  Avg log-area: 1.17

  Full means:
            FFMC     DMC      DC    ISI   temp     RH  wind  rain   area  \
cluster                                                                    
Cluster 1  86.38   31.09  110.25   5.66  11.76  46.50  4.74  0.00   5.98   
Cluster 2  91.40  130.45  646.28  10.98  18.49  52.27  5.49  0.07   9.86   
Cluster 3  91.96  132.39  671.05   9.29  22.01  38.93  2.90  0.00  17.31   

           log_area  
cluster              
Cluster 1      0.99  
Cluster 2      1.09  
Cluster 3  

---
## 6. Do the Clusters Differ in Fire Size?

The clustering above used **zero information about fire size**. It only saw weather and drought conditions. Yet when we now look at fire size within each cluster, the pattern seems meaningful.


### Visual 2 —  Environmental Conditions vs Average Fire Size

Each grouped bar shows one cluster's average value for temperature, DC, DMC, and burned area. The right panel shows the proportion of fires in each cluster that spread at all. The **legend** toggles variables on and off.

In [7]:
clustered = fit_clusters(weight=2.0, k=3)
bridge = clustered.groupby('cluster').agg(
    n         = ('area', 'count'),
    avg_area  = ('area', 'mean'),
    avg_temp  = ('temp', 'mean'),
    avg_DC    = ('DC',   'mean'),
    avg_wind  = ('wind', 'mean'),
    pct_spread= ('area', lambda x: (x > 0).mean() * 100)
).round(1).reset_index()
bridge['label'] = bridge['cluster'].map(CLUST_NAMES)
bridge['color'] = bridge['cluster'].map(CLUST_COLORS)

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Environmental Conditions per Cluster', 'Fire Spread Rate per Cluster'),
    column_widths=[0.60, 0.40],
    horizontal_spacing=0.12
)

bar_metrics = [
    ('avg_temp', 'Avg Temp (°C)',  '#DC2626'),
    ('avg_DC',   'Avg DC (÷10)',   '#EA580C'),
    ('avg_wind', 'Avg Wind (km/h)','#F59E0B'),
    ('avg_area', 'Avg Area (ha)',  '#92400E'),
]

for col, label, bcolor in bar_metrics:
    vals = bridge['avg_DC'] / 10 if col == 'avg_DC' else bridge[col]
    fig2.add_trace(go.Bar(
        x=bridge['label'], y=vals, name=label,
        marker_color=bcolor, opacity=0.85,
        text=vals.round(1), textposition='outside',
        hovertemplate=f'<b>%{{x}}</b><br>{label}: %{{y:.1f}}<extra></extra>'
    ), row=1, col=1)

for _, row in bridge.iterrows():
    fig2.add_trace(go.Bar(
        x=[row['label']], y=[row['pct_spread']],
        name=row['label'], marker_color=row['color'],
        showlegend=False,
        text=[f"{row['pct_spread']}%"], textposition='outside',
        hovertemplate=f'<b>{row["label"]}</b><br>Spread rate: {row["pct_spread"]}%<extra></extra>'
    ), row=1, col=2)

fig2.update_layout(
    title=dict(
        text='<b>Visual 2:</b> Clusters (found without fire size) differ in fire size',
        font=dict(size=14, color='#2C1810'),
        x=0.5, xanchor='center'
    ),
    template='plotly_white',
    paper_bgcolor='white',
    plot_bgcolor='white',
    height=500,
    barmode='group',
    font=dict(family='Georgia, serif', size=12, color='#2C1810'),
    margin=dict(t=100, b=60, l=60, r=40),
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.08,
        xanchor='center', x=0.5
    )
)

fig2.update_yaxes(title_text='Value', row=1, col=1, range=[0, 85],
                  gridcolor='#F0E8E0', linecolor='#D4C4B8')
fig2.update_yaxes(title_text='% of Fires', row=1, col=2, range=[0, 75],
                  gridcolor='#F0E8E0', linecolor='#D4C4B8')
fig2.update_xaxes(tickangle=-15, row=1, col=1, linecolor='#D4C4B8')
fig2.update_xaxes(tickangle=-15, row=1, col=2, linecolor='#D4C4B8')

fig2.show()

---
## 7. Stage 2 — Can a Model Predict Fire Size from Conditions?

Stage 1 showed that conditions naturally separate fires into meaningful burned area groups. Stage 2 asks whether a supervised model can *learn* to predict fire size from those same conditions and whether a flexible nonlinear model does better than a linear one.

We now work with the 270 fires that spread (area > 0) and classify each as **large** (above median log-area) or **small** (below median). Three models are compared:
- **Logistic Regression** — assumes linear boundaries between classes
- **Decision Tree** — nonlinear but shallow (max depth 5)
- **Random Forest** — 200-tree ensemble, can capture complex interactions


### Visual 3 — H1, H2 & H3: Key Predictors vs Log-Transformed Fire Size

The **H1 / H2 / H3 buttons**  switch between temperature, drought code, and wind speed. Dashed trendlines show the direction of each relationship for large and small fires separately.

In [8]:
features_num = ['temp', 'DC', 'wind']

X = df[features_num]
y = df['large_fire']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([('num', StandardScaler(), features_num)])
preprocessor_raw = ColumnTransformer([('num', 'passthrough', features_num)])

log_model = Pipeline([('pre', preprocessor),     ('model', LogisticRegression(max_iter=1000, random_state=42))])
dt_model  = Pipeline([('pre', preprocessor_raw), ('model', DecisionTreeClassifier(max_depth=5, random_state=42))])
rf_model  = Pipeline([('pre', preprocessor_raw), ('model', RandomForestClassifier(n_estimators=200, random_state=42))])

models  = {'Logistic Regression': log_model, 'Decision Tree': dt_model, 'Random Forest': rf_model}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results.append({'Model':     name,
        'Accuracy':  accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall':    recall_score(y_test, preds),
        'F1':        f1_score(y_test, preds),
        'AUC':       roc_auc_score(y_test, proba)})

results_df = pd.DataFrame(results)
print(results_df.round(3).to_string(index=False))

              Model  Accuracy  Precision  Recall    F1   AUC
Logistic Regression     0.574      0.600   0.444 0.511 0.488
      Decision Tree     0.481      0.481   0.481 0.481 0.502
      Random Forest     0.611      0.636   0.519 0.571 0.617


In [9]:
fire_label = df['large_fire'].map({0: 'Small Fire', 1: 'Large Fire'})

PREDICTORS = [
    ('temp', 'Temperature (°C)',  'Temperature vs Log-Transformed Burned Area'),
    ('DC',   'Drought Code (DC)', 'Drought Code vs Log-Transformed Burned Area'),
    ('wind', 'Wind Speed (km/h)', 'Wind Speed vs Log-Transformed Burned Area'),
]

def ols_line(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.array([x.min(), x.max()]), np.array([y.mean(), y.mean()])
    m, b = np.polyfit(x[mask], y[mask], 1)
    xr   = np.array([x[mask].min(), x[mask].max()])
    return xr, m * xr + b

fig3       = go.Figure()
TRACES_PER = 4

for p_idx, (col, xlabel, title) in enumerate(PREDICTORS):
    visible = (p_idx == 0)
    for label, colour in [('Small Fire', C_LOW), ('Large Fire', C_HIGH)]:
        sub = df[fire_label == label]
        fig3.add_trace(go.Scatter(
            x=sub[col], y=sub['log_area'], mode='markers',
            name=label, marker=dict(color=colour, opacity=0.60, size=7),
            legendgroup=label, showlegend=(p_idx == 0), visible=visible,
            hovertemplate=f'<b>{label}</b><br>{xlabel}: %{{x:.1f}}<br>Log Area: %{{y:.3f}}<extra></extra>'
        ))
        xr, yr = ols_line(sub[col].values.astype(float), sub['log_area'].values.astype(float))
        fig3.add_trace(go.Scatter(
            x=xr, y=yr, mode='lines', name=f'{label} trend',
            line=dict(color=colour, width=2.5, dash='dash'),
            legendgroup=label, showlegend=False, visible=visible, hoverinfo='skip'
        ))

total_traces = len(PREDICTORS) * TRACES_PER
buttons = []
for p_idx, (col, xlabel, title) in enumerate(PREDICTORS):
    vis = [False] * total_traces
    for t in range(TRACES_PER):
        vis[p_idx * TRACES_PER + t] = True
    buttons.append(dict(
        label=xlabel,
        method='update',
        args=[
            {'visible': vis},
            {
                'title': {'text': f'<b>Visual 3:</b> {title}', 'font': {'size': 14, 'color': C_TITLE}},
                'xaxis': {'title': {'text': xlabel}},
                'annotations': []
            }
        ]
    ))

fig3.update_layout(
    title=dict(
        text=f'<b>Visual 3:</b> {PREDICTORS[0][2]}',
        font=dict(size=14, color=C_TITLE),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(title=PREDICTORS[0][1]),
    yaxis=dict(title='Log-Transformed Burned Area'),
    template='plotly_white',
    paper_bgcolor='white',
    plot_bgcolor='#FFF8F5',
    height=500,
    margin=dict(t=90, b=60, l=70, r=40),
    updatemenus=[dict(
        type='buttons', direction='right',
        x=0.5, y=1.12,
        xanchor='center', yanchor='top',
        buttons=buttons,
        bgcolor='white', bordercolor='#EA580C',
        font=dict(size=12, color='#2C1810')
    )],
    legend=dict(title='Fire Size', x=1.01, y=1, xanchor='left')
)
fig3.show()

### Visual 4 — ROC Curves: Gow good are the models?

The ROC curve measures how well each model separates large from small fires across every possible decision threshold. AUC = 0.50 is random chance, a model that learned nothing. AUC = 1.0 is perfect.

In [10]:
ROC_COLORS = {
    'Logistic Regression': '#F59E0B',
    'Decision Tree':       '#EA580C',
    'Random Forest':       '#DC2626'
}
FILL_COLORS = {
    'Logistic Regression': 'rgba(245,158,11,0.15)',
    'Decision Tree':       'rgba(234,88,12,0.15)',
    'Random Forest':       'rgba(220,38,38,0.18)'
}

fig4 = go.Figure()

for name, model in models.items():
    proba            = model.predict_proba(X_test)[:, 1]
    fpr, tpr, thresh = roc_curve(y_test, proba)
    auc              = roc_auc_score(y_test, proba)
    fig4.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'{name}  (AUC = {auc:.3f})',
        line=dict(color=ROC_COLORS[name], width=3),
        fill='tozeroy', fillcolor=FILL_COLORS[name],
        hovertemplate=f'<b>{name}</b><br>FPR: %{{x:.3f}}<br>TPR: %{{y:.3f}}<br>AUC: {auc:.3f}<extra></extra>'
    ))

fig4.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Random Chance (AUC = 0.500)',
    line=dict(color='#A8A29E', dash='dash', width=2),
    hoverinfo='skip'
))

fig4.update_layout(
    title=dict(
        text='<b>Visual 4:</b> ROC Curves — Model Comparison',
        font=dict(size=14, color=C_TITLE),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='False Positive Rate',
        range=[-0.02, 1.02],
        gridcolor='#F0E8E0',
        linecolor='#D4C4B8'
    ),
    yaxis=dict(
        title='True Positive Rate',
        range=[-0.02, 1.05],
        gridcolor='#F0E8E0',
        linecolor='#D4C4B8'
    ),
    template='plotly_white',
    paper_bgcolor='white',
    plot_bgcolor='#FFF8F5',
    height=500,
    margin=dict(t=80, b=60, l=70, r=40),
    font=dict(family='Georgia, serif', size=12, color='#2C1810'),
    legend=dict(
        x=0.58, y=0.08,
        bgcolor='rgba(255,255,255,0.95)',
        bordercolor='#D4C4B8',
        borderwidth=1
    )
)

fig4.show()

### Visual 5 — Feature Importance

Random Forest feature importances measure how much each variable reduced uncertainty across all 200 trees. This directly tests H1, H2, and H3 and lets us check whether the classifier learned the same structure the clustering revealed.

In [11]:
importances = rf_model.named_steps['model'].feature_importances_

imp_df = pd.DataFrame({
    'Feature':    ['Temperature', 'Drought Code', 'Wind Speed'],
    'Importance': [importances[0], importances[1], importances[2]],
    'Color':      [C_HIGH, C_MOD, C_LOW],
}).sort_values('Importance', ascending=True)

fig5 = go.Figure(go.Bar(
    x=imp_df['Importance'], y=imp_df['Feature'],
    orientation='h',
    marker_color=imp_df['Color'],
    text=[f"{v:.3f}" for v in imp_df['Importance']],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Importance: %{x:.3f}<extra></extra>'
))

fig5.update_layout(
    title=dict(
        text='<b>Visual 5:</b> Feature Importance — Random Forest',
        font=dict(size=14, color=C_TITLE),
        x=0.5, xanchor='center'
    ),
    xaxis=dict(title='Mean Gini Importance', range=[0, 0.50]),
    template='plotly_white',
    paper_bgcolor='white',
    plot_bgcolor='#FFF8F5',
    height=350,
    margin=dict(t=80, b=60, l=120, r=60),
    font=dict(family='Georgia, serif', size=12, color='#2C1810')
)
fig5.show()

---
## 8. Hypothesis Evaluation

H1: Temperature predicts fire size — Supported. Higher temperatures were associated with larger fires, with temperature being the top cluster separator (11.8°C → 23.3°C) and the top feature in importance at 0.352.

H2: Drought Code predicts fire size — Supported. Higher DC values were associated with larger fires, creating the largest cluster gap (DC 118 vs 691) and ranking 2nd in feature importance at 0.320.

H3: Wind speed predicts fire size — Partially Supported. Wind ranked 3rd in feature importance at 0.250 but showed weaker cluster separation compared to temperature and DC.

H4: Random Forest beats linear models — Supported. The Random Forest achieved an AUC of 0.625 compared to 0.480 for Logistic Regression, consistent with the curved cluster boundaries observed in the data.

---

## 9. Conclusion 

**Research question:** *Do environmental conditions like DC, tempm and wind speed, encode enough signal to predict whether a fire will be large or small?*

**Answer: Yes — meaningfully, but not completely.**

An AUC of 0.625 reflects signal extracted from genuinely noisy data.

**Limitations:**
- Only 270 fires with measurable spread — a small sample for machine learning
- Dataset covers one park in Portugal; generalisability is unknown
- DC and month are partially correlated (drought accumulates over summer)
- Factors not measured here such as terrain, fuel moisture, suppression — likely explain much of the residual variation

**Future work:** Adding terrain slope, fuel load maps, and a larger multi-region dataset would likely push the AUC substantially higher and produce cluster boundaries that align more tightly with fire size outcomes.

# Making the Dashboard

In [12]:
from plotly.io import to_html

interactive_fig = fig

top = to_html(interactive_fig, full_html=False, include_plotlyjs='cdn')

v1 = to_html(fig1, full_html=False, include_plotlyjs=False)
v2 = to_html(fig2, full_html=False, include_plotlyjs=False)
v3 = to_html(fig3, full_html=False, include_plotlyjs=False)
v4 = to_html(fig4, full_html=False, include_plotlyjs=False)
v5 = to_html(fig5, full_html=False, include_plotlyjs=False)

html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Wildfire Severity Analysis</title>
    <link href="https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700;900&family=Crimson+Pro:ital,wght@0,300;0,400;1,300&display=swap" rel="stylesheet">

    <style>
        :root {{
            --bg:           #FAFAF8;
            --card-bg:      #FFFFFF;
            --card-border:  #E8DDD5;
            --text:         #2C1810;
            --text-muted:   #8C7B72;
            --red:          #DC2626;
            --orange:       #EA580C;
            --amber:        #F59E0B;
        }}

        * {{ box-sizing: border-box; margin: 0; padding: 0; }}

        body {{
            font-family: 'Crimson Pro', Georgia, serif;
            background-color: var(--bg);
            min-height: 100vh;
            color: var(--text);
        }}

        .hero {{
            text-align: center;
            padding: 52px 24px 36px;
            border-bottom: 1px solid var(--card-border);
            margin-bottom: 32px;
        }}

        .hero h1 {{
            font-family: 'Playfair Display', Georgia, serif;
            font-size: clamp(22px, 3vw, 40px);
            font-weight: 700;
            line-height: 1.25;
            color: var(--text);
            margin-bottom: 12px;
        }}

        .hero .subtitle {{
            font-size: 16px;
            font-style: italic;
            color: var(--text-muted);
        }}

        .section-label {{
            max-width: 1100px;
            margin: 0 auto 12px;
            padding: 0 20px;
            display: flex;
            align-items: center;
            gap: 12px;
        }}

        .section-label .badge {{
            background: var(--orange);
            color: white;
            font-size: 11px;
            font-weight: 700;
            letter-spacing: 1.5px;
            text-transform: uppercase;
            padding: 4px 12px;
            border-radius: 4px;
        }}

        .section-label .label-text {{
            font-size: 13px;
            color: var(--text-muted);
            font-style: italic;
        }}

        .chart-box {{
            background: var(--card-bg);
            border: 1px solid var(--card-border);
            border-radius: 12px;
            padding: 20px;
            margin-bottom: 24px;
            max-width: 1100px;
            margin-left: auto;
            margin-right: auto;
            box-shadow: 0 2px 8px rgba(0,0,0,0.06);
        }}

        .chart-box-centered {{
            background: var(--card-bg);
            border: 1px solid var(--card-border);
            border-radius: 12px;
            padding: 20px;
            margin-bottom: 24px;
            max-width: 1100px;
            margin-left: auto;
            margin-right: auto;
            display: flex;
            justify-content: center;
            box-shadow: 0 2px 8px rgba(0,0,0,0.06);
        }}

        .footer {{
            text-align: center;
            padding: 28px 24px;
            color: var(--text-muted);
            font-size: 13px;
            border-top: 1px solid var(--card-border);
            margin-top: 20px;
        }}

        .updatemenu-button rect {{ fill: white !important; }}
        .updatemenu-button.active rect,
        .updatemenu-button:active rect {{ fill: #FFF8F5 !important; }}
        .updatemenu-button text {{ fill: #EA580C !important; }}

        ::-webkit-scrollbar {{ width: 6px; }}
        ::-webkit-scrollbar-track {{ background: var(--bg); }}
        ::-webkit-scrollbar-thumb {{ background: var(--orange); border-radius: 3px; }}

        .container {{ padding: 0 20px 20px; }}
    </style>
</head>

<body>

<div class="hero">
        <h1>Do Temperature, Drought, and Wind<br>Predict Wildfire Severity?</h1>
        <p class="subtitle">Montesinho Natural Park, Portugal — 517 fire records, two-stage ML analysis</p>
    </div>

    <div class="container">

        <div class="section-label">
            <span class="badge">Explorer</span>
            <span class="label-text">Toggle between environmental drivers</span>
        </div>
        <div class="chart-box-centered">{top}</div>

        <div class="section-label">
            <span class="badge">Visual 1</span>
            <span class="label-text">Unsupervised clustering in PCA space — drag the weight slider</span>
        </div>
        <div class="chart-box-centered">{v1}</div>

        <div class="section-label">
            <span class="badge">Visual 2</span>
            <span class="label-text">Do the clusters differ in fire severity?</span>
        </div>
        <div class="chart-box">{v2}</div>

        <div class="section-label">
            <span class="badge">Visual 3</span>
            <span class="label-text">Small vs large fire scatter plots by predictor</span>
        </div>
        <div class="chart-box">{v3}</div>

        <div class="section-label">
            <span class="badge">Visual 4</span>
            <span class="label-text">ROC curve comparison across all classifiers</span>
        </div>
        <div class="chart-box">{v4}</div>

        <div class="section-label">
            <span class="badge">Visual 5</span>
            <span class="label-text">Which features matter most to Random Forest?</span>
        </div>
        <div class="chart-box">{v5}</div>

    </div>

    <div class="footer">
        <strong>Wildfire Severity Analysis</strong> &nbsp;·&nbsp; Montesinho Natural Park, Portugal
    </div>

</body>
</html>
"""

with open('wildfire_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)

print('Dashboard saved to wildfire_dashboard.html')

Dashboard saved to wildfire_dashboard.html
